# Analytic fixed boundary → PF currents → free boundary (#1608)

This executed example runs **20 native cases**: Solov'ev/Cerfon–Freidberg, Guazzotto–Freidberg Part 1, and a representable current pedestal, each in limited, lower-single-null and double-null geometry through both the direct Green and TokaMaker `get_vfixed()` inverse routes; it adds pure-pressure limited Guazzotto cases through both routes. Every case performs the initial fixed-current free solve, optional shape refinement, and a second unconstrained free solve with the *final PF currents frozen*. The public `fixed_to_free` API performs the inverse and native solves; this notebook does not reimplement them.

The outputs below were produced with an installed OpenFUSIONToolkit (OFT) on an isolated vestserver run. Set `VAFT_RUN_NATIVE=1`, `VAFT_1608_OUT` to a **new** directory, and `VAFT_1608_SOURCE_COMMIT` to the source commit to rerun them. Raw solver logs, meshes and g-files stay in that directory; this notebook stores compact numerical tables and figures. Without the opt-in variables, it explains how to run and does not start OFT. The ±200 kA bounds below describe a synthetic 12-coil machine, **not VEST hardware ratings**.

In [ ]:
import os
from contextlib import redirect_stderr, redirect_stdout
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from vaft.code.tokamaker import TokaMakerConfig, fixed_to_free
from vaft.process.equilibrium import as_equilibrium, derive_boundary_representation
from vaft.validation.studies.fixed_free_1608.matrix import (
    acceptance_failures, machine_case, target_case,
)

RUN_NATIVE = os.environ.get('VAFT_RUN_NATIVE') == '1'
SOURCE_COMMIT = os.environ.get('VAFT_1608_SOURCE_COMMIT', 'not recorded')
print('Source commit:', SOURCE_COMMIT)
print('Native execution:', 'enabled' if RUN_NATIVE else 'disabled; set VAFT_RUN_NATIVE=1 and VAFT_1608_OUT')
print('Target grid: 65 x 65; plasma mesh dx: 0.04 m; vacuum dx: 0.08 m; coil dx: 0.02 m')
print('Machine: 12 independent one-turn PF coils, ±200 kA each; initial and refinement regularization: 1e-5')

## Case matrix and physical separation

The direct route computes the target plasma self-field with the existing exact axisymmetric Green response. The native route gets its external vacuum target from an independent TokaMaker fixed-boundary solve and `get_vfixed()`. Both use the same synthetic PF geometry. `fit.accepted` refers to the *initial linear inverse problem*; it is never upgraded by subsequent free-boundary convergence. The final benchmark gate applies only to the separate, frozen-current physical closure. For diverted cases the direct route supplies explicit target X-point saddle constraints. COCOS 11 full-weber flux and physical circuit amperes are used by VAFT; native FEM `get_vfixed()` samples are converted with +2π.

In [ ]:
FAMILIES = ('solovev', 'guazzotto_freidberg', 'guazzotto_pedestal')
TOPOLOGIES = ('limited', 'lower_single_null', 'double_null')
BACKENDS = ('direct', 'tokamaker_vfixed')
CASES = [(family, topology, .5, .1 if family == 'guazzotto_pedestal' else 0.)
         for family in FAMILIES for topology in TOPOLOGIES]
CASES.append(('guazzotto_freidberg', 'limited', 1., 0.))  # pure pressure
assert len(CASES) * len(BACKENDS) == 20

bounds = {f'PF{i+1}': (-2e5, 2e5) for i in range(12)}
records = []
contours = {}
currents = {}
if RUN_NATIVE:
    workdir = Path(os.environ['VAFT_1608_OUT']).expanduser().resolve()
    workdir.mkdir(parents=True, exist_ok=False)
    logdir = workdir / '_logs'
    logdir.mkdir()
    print('Fresh isolated run directory created; individual paths are kept out of saved outputs.')

## Run all 20 cases

Each call below invokes the **public** `fixed_to_free` bridge. The native solver's detailed stdout and files are retained beside its case directory. A failure is recorded for its own case and does not erase the other cases. The notebook's saved output is intentionally concise; inspect the isolated logs for detailed native diagnostics.

In [ ]:
if RUN_NATIVE:
    for family, topology, nu, pedestal in CASES:
        target = target_case(family, topology, pedestal=pedestal, nu=nu, resolution=65)
        target, machine, wall = machine_case(target, topology)
        case_name = f'{family}_{topology}_nu{nu:g}'
        contours[case_name] = {'target': (target.lcfs.r.copy(), target.lcfs.z.copy())}
        for backend in BACKENDS:
            name = f'{case_name}_{backend}'
            record = {'family': family, 'topology': topology, 'nu': nu,
                      'backend': backend, 'case': case_name, 'name': name,
                      'error': None}
            config = TokaMakerConfig(profile_mode='equilibrium', dx_plasma=.04,
                                     dx_vacuum=.08, dx_coil=.02, limiter=wall,
                                     lim_zmax=None, neck_limiter_zmax=None, nthreads=2)
            fit_options = {'regularization': 1e-5, 'current_bounds': bounds}
            if backend == 'direct':
                x_points = [(x.r, x.z) for x in derive_boundary_representation(target).x_points if x.active]
                fit_options.update(method='flux_normal', boundary_samples=64,
                                   x_points=x_points)
            try:
                with (logdir / f'{name}.log').open('w') as solver_log:
                    with redirect_stdout(solver_log), redirect_stderr(solver_log):
                        result = fixed_to_free(
                            target, machine, workdir / name,
                            fit_backend=backend, config=config,
                            fit_options=fit_options,
                            refine_shape=True, verify_refinement=True)
                record.update(
                    fit_status=result.fit.status,
                    fit_accepted=result.fit.accepted,
                    flux_rms=result.fit.rms_relative_flux,
                    flux_max=result.fit.max_relative_flux,
                    condition=result.fit.condition_number,
                    initial_status=result.status,
                    refinement_status=result.refinement_status,
                    verification_status=result.verification_status,
                    current_change_A=result.refinement_diagnostics.get('max_current_change_A'),
                    verified_comparison=result.verified_comparison,
                    initial_error=result.forward.error,
                    refined_error=result.refined.error if result.refined else None,
                    verification_error=result.verified_free.error if result.verified_free else None,
                )
                currents[(case_name, backend)] = {
                    'initial': dict(result.initial_currents_A),
                    'final': dict(result.refined_currents_A)}
                if result.verified_free is not None and result.verified_free.ok:
                    solved = as_equilibrium(result.verified_free.gfile, convention=7)
                    contours[case_name][backend] = (solved.lcfs.r.copy(), solved.lcfs.z.copy())
            except Exception as error:
                record['error'] = f'{type(error).__name__}: {error}'
            record['gate_failures'] = acceptance_failures({
                'topology': topology,
                'error': record['error'],
                'verification_status': record.get('verification_status'),
                'verified_comparison': record.get('verified_comparison'),
            })
            record['gate_accepted'] = not record['gate_failures']
            records.append(record)
            print(f'{name}: fit={record.get("fit_status", "error")}, '
                  f'initial={record.get("initial_status", "error")}, '
                  f'refined={record.get("refinement_status", "error")}, '
                  f'frozen={record.get("verification_status", "error")}, '
                  f'gate={"pass" if record["gate_accepted"] else "fail"}')
else:
    print('Native solve skipped. The committed tables and figures below show the validated server run.')

## Fit, solve, and physics comparison

These rows preserve four separate decisions. An initial inverse fit can be rejected even when shape refinement later finds a good coil-driven equilibrium. A failed initial fixed-current solve remains failed. `frozen` is the unconstrained second free solve with the refined PF currents fixed. The benchmark gate checks that final physical result; it does not certify hardware ratings beyond the synthetic bounds.

In [ ]:
def metric(record, *keys):
    value = record.get('verified_comparison') or {}
    for key in keys:
        value = value.get(key, {}) if isinstance(value, dict) else {}
    return value if isinstance(value, (int, float)) and np.isfinite(value) else None

def fmt(value, digits=3, scale=1.):
    return '—' if value is None else f'{value*scale:.{digits}f}'

if records:
    lines = ['| Family / topology / ν | Route | Fit | Flux RMS / max | Cond. | Initial | Refined | Frozen | ΔI max [kA] | LCFS max [mm] | Axis [mm] | X max [mm] | Δq95 | ΔIp [A] | Gate |',
             '| --- | --- | --- | ---: | ---: | --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |']
    for rec in records:
        x = (rec.get('verified_comparison') or {}).get('native_boundary', {}).get('matched_displacements_m') or []
        lines.append('| ' + ' | '.join([
            f"{rec['family']} / {rec['topology']} / {rec['nu']:g}",
            rec['backend'], rec.get('fit_status', 'error'),
            f"{fmt(rec.get('flux_rms'), 4)} / {fmt(rec.get('flux_max'), 4)}",
            fmt(rec.get('condition'), 1),
            rec.get('initial_status', 'error'), rec.get('refinement_status', 'error'),
            rec.get('verification_status', 'error'),
            fmt(rec.get('current_change_A'), 2, .001),
            fmt(metric(rec, 'lcfs', 'max_distance_m'), 2, 1000.),
            fmt(metric(rec, 'axis_displacement_m'), 2, 1000.),
            fmt(max(x) if x else 0., 3, 1000.),
            fmt(metric(rec, 'globals', 'q95_field_integral', 'difference'), 4),
            fmt(metric(rec, 'globals', 'Ip_A', 'difference'), 2),
            'pass' if rec['gate_accepted'] else 'fail',
        ]) + ' |')
    display(Markdown('\n'.join(lines)))
    print(f"Frozen-current benchmark: {sum(r['gate_accepted'] for r in records)}/{len(records)} passed")
    for rec in records:
        if not rec['gate_accepted']:
            print(rec['name'], ':', '; '.join(rec['gate_failures']))
else:
    print('No live results in this kernel. Enable native execution to regenerate the stored table.')

## Target and frozen-current LCFS

Each panel overlays the prescribed analytic boundary with the unconstrained free-boundary result from both fitting routes. The X-point topology gate uses target-matched native finite-element saddle diagnostics; coarse g-file contours can miss a narrow X-point. The plot is geometry evidence, not the sole topology test.

In [ ]:
if records:
    fig, axes = plt.subplots(5, 2, figsize=(10, 18), sharex=True, sharey=True)
    for ax, (family, topology, nu, pedestal) in zip(axes.flat, CASES):
        case_name = f'{family}_{topology}_nu{nu:g}'
        curves = contours[case_name]
        ax.plot(*curves['target'], color='black', linewidth=1.4, label='analytic target')
        for backend, color in [('direct', 'tab:blue'), ('tokamaker_vfixed', 'tab:orange')]:
            if backend in curves:
                ax.plot(*curves[backend], color=color, linewidth=1.,
                        linestyle='--' if backend == 'tokamaker_vfixed' else '-',
                        label=backend)
        ax.set_title(f'{family} · {topology} · ν={nu:g}', fontsize=9)
        ax.set_aspect('equal')
        ax.grid(alpha=.2)
        ax.set_xlabel('R [m]')
        ax.set_ylabel('Z [m]')
    axes[0, 0].legend(fontsize=7)
    fig.tight_layout()
    plt.show()

## Initial and final PF-current vectors

Each panel shows all twelve physical circuit currents in amperes. The two inverse routes need not return the same current vector: near-redundant coil responses make boundary-field and frozen equilibrium agreement the meaningful physical comparisons. Current changes caused by shape refinement remain visible.

In [ ]:
if records:
    fig, axes = plt.subplots(5, 2, figsize=(11, 15), sharex=True)
    coil_names = [f'PF{i+1}' for i in range(12)]
    x = np.arange(12)
    for ax, (family, topology, nu, pedestal) in zip(axes.flat, CASES):
        case_name = f'{family}_{topology}_nu{nu:g}'
        for backend, color in [('direct', 'tab:blue'), ('tokamaker_vfixed', 'tab:orange')]:
            series = currents.get((case_name, backend))
            if not series:
                continue
            for stage, style in [('initial', ':'), ('final', '-')]:
                values = [series[stage].get(name, float('nan')) / 1000 for name in coil_names]
                ax.plot(x, values, color=color, linestyle=style, linewidth=1.,
                        marker='.', markersize=3, label=f'{backend} {stage}')
        ax.set_title(f'{family} · {topology} · ν={nu:g}', fontsize=9)
        ax.set_ylabel('Current [kA]')
        ax.grid(alpha=.2)
    for ax in axes[-1]:
        ax.set_xticks(x, coil_names, rotation=45, fontsize=7)
        ax.set_xlabel('PF circuit')
    axes[0, 0].legend(fontsize=6, ncol=2)
    fig.tight_layout()
    plt.show()

## Interpretation and reproducibility

The two inverse targets are constructed independently: the direct route integrates the target plasma source, while the native route uses TokaMaker's fixed-boundary `get_vfixed()` vacuum target. All reported PF currents are physical amperes. The 20-case matrix includes cases where the initial fixed-current solve fails; later refinement and frozen-current convergence do not relabel those initial outcomes. The synthetic machine is a benchmark, not a statement that these currents satisfy VEST engineering limits.

The source and environment are printed above. Every rerun needs a fresh `VAFT_1608_OUT`; raw native artifacts and per-case logs remain there. The companion [`measured_matrix.json`](../vaft/validation/studies/fixed_free_1608/measured_matrix.json) contains the earlier independent server benchmark. This notebook's stored outputs come from its **own** execution, not that JSON. The repository's output cleaner removes execution counters, machine paths, widget state and unsupported MIME types while retaining these static Markdown and PNG outputs.